# MuseGAN: Generating Short Polyphonic Phrases

## Case study: multi-track pianorolls in the style of Bach Chorales

MuseGAN is a family of generative adversarial networks designed for multi-track polyphonic music. In the original formulation, the generator creates several synchronized parts (for example, instruments), and a critic evaluates their plausibility. This notebook builds a small educational variant inspired by MuseGAN: the generator maps a noise vector to a pianoroll with `[time, pitch]` axes, and the critic distinguishes real examples from synthetic ones.

### Why use a GAN for music?

A GAN trains a generator $G(z)$ to match the data distribution without an explicit likelihood. A critic $D(x)$ receives real and generated examples, and the gradient of its score guides the generator toward the data distribution. A standard GAN uses a minimax game; here we use hinge loss and spectral normalization in the critic for a more stable demonstration.

### Polyphony, bars, and representation

A multi-part score is not a single stream of notes. The parts need to preserve rhythmic alignment and harmony. A pianoroll represents notes as a binary time-by-pitch matrix; extending it to `[track, bar, time, pitch]` gives a natural multi-track format. For simplicity, our miniature MuseGAN emits one shared pianoroll with several active notes per step.

### Adversarial training and common problems

The critic is updated on real and fake parts; the generator is updated so the critic assigns its parts a high score. Differences in update counts, learning rates, and network capacity can destabilize the game. Mode collapse means that many noise vectors produce similar outputs. In this small example, we inspect note activity and sample diversity rather than promise high musical quality.

Original MuseGAN variants use, among other things, multi-track generators, latent codes shared across bars and tracks, and temporal modules that generate continuity. The model below is deliberately smaller and is not a faithful reproduction of the published architecture.

### Data and limitations of the demonstration

The shared loader attempts to download JSB Chorales and generates a clearly labeled synthetic harmony corpus when the data is unavailable. The critic sees short pianoroll fragments. A short run is feasible on a CPU, but adversarial quality usually requires more data, tuning, checkpoints, and listening-based evaluation. Do not treat loss plots as a standalone quality ranking.

## 1. Data: JSB Chorales and the pianoroll representation

Bach Chorales is a classic small dataset of symbolic four-part scores. Each time step describes the pitches in the soprano, alto, tenor, and bass voices. In this case study, we use a simplified pianoroll representation: a `[time, pitch]` tensor where 1 indicates an active note. The loader first tries to download a CSV file from the JSB Chorales dataset. If the network or source URL is unavailable, it generates deterministic example harmony sequences; the cell reports which option was used. This lets you follow the full pipeline while making the demonstration fallback limitations explicit.

For the real dataset, we assume a CSV with cells containing lists of MIDI notes active at successive time steps. For stability, the example maps notes to 48 pitch classes (MIDI 48–95) and sequences to a fixed length. This is a simplification: note durations, dynamics, articulation, and precise timing are not encoded.

**What this cell does:** imports dependencies, sets random seeds, and builds a small example corpus of multi-voice pianorolls. You can enable an attempt to download JSB by setting `TRY_DOWNLOAD=True`.


In [ ]:
import math, random, urllib.request, csv, io
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

SEED=23
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:",device)

PITCH_MIN, N_PITCH, STEPS = 48, 48, 32
TRY_DOWNLOAD=False  # Set to True to try downloading the public dataset.

def make_demo_corpus(n=240):
    # Four voices, diatonic progressions, and sustained rhythms: a controlled fallback.
    scales=[0,2,4,5,7,9,11]
    roots=[0,5,7,0,9,5,0,7]
    rows=[]
    for _ in range(n):
        roll=np.zeros((STEPS,N_PITCH),dtype=np.float32)
        trans=random.choice([-2,0,0,0,2])
        for t in range(STEPS):
            beat=t//4; root=roots[beat%len(roots)]
            chord=[root,(root+4)%12,(root+7)%12,(root+11)%12]
            pitches=[60+root+trans, 60+chord[1]+trans, 48+chord[2]+trans, 48+root+trans]
            for voice,p in enumerate(pitches):
                p=max(PITCH_MIN,min(PITCH_MIN+N_PITCH-1,p))
                # Octave/pitch variation and occasional rests imitate variations between parts.
                q=p + (12 if voice<2 and random.random()<.25 else 0)
                q=max(PITCH_MIN,min(PITCH_MIN+N_PITCH-1,q))
                if random.random()>.035: roll[t,q-PITCH_MIN]=1
        rows.append(roll)
    return np.stack(rows)

def load_jsb_or_demo():
    if TRY_DOWNLOAD:
        urls=["https://raw.githubusercontent.com/czhuang/JSB-Chorales-dataset/master/jsb-chorales-train.csv"]
        for url in urls:
            try:
                raw=urllib.request.urlopen(url,timeout=8).read().decode("utf-8")
                # This public format stores one observation per row.
                parsed=[]; sequence=[]
                for row in csv.reader(io.StringIO(raw)):
                    vals=[int(x) for x in row if x.strip().lstrip('-').isdigit()]
                    if not vals:
                        if len(sequence)>=STEPS:
                            for start in range(0,len(sequence)-STEPS+1,STEPS//2):
                                parsed.append(np.stack(sequence[start:start+STEPS]))
                        sequence=[]; continue
                    frame=np.zeros(N_PITCH,dtype=np.float32)
                    for p in vals:  # Multiple voices/notes may be active in one time step.
                        if PITCH_MIN<=p<PITCH_MIN+N_PITCH: frame[p-PITCH_MIN]=1
                    sequence.append(frame)
                if len(sequence)>=STEPS:
                    for start in range(0,len(sequence)-STEPS+1,STEPS//2):
                        parsed.append(np.stack(sequence[start:start+STEPS]))
                if len(parsed)>20: return np.stack(parsed),"downloaded JSB Chorales (CSV; polyphonic time steps)"
            except Exception as e: print("Download failed:",type(e).__name__,str(e)[:120])
    return make_demo_corpus(),"demonstration fallback (synthetic harmonies)"

rolls,data_source=load_jsb_or_demo()
print("Source:",data_source,"| number of sequences:",len(rolls),"| shape:",rolls.shape)
plt.figure(figsize=(10,2.5)); plt.imshow(rolls[0].T,aspect="auto",origin="lower",cmap="magma"); plt.xlabel("step"); plt.ylabel("MIDI pitch (index)" ); plt.title("Example pianoroll"); plt.show()


## 2. Data as tensors of real phrases

The critic works with fixed-length pianorolls. Each example contains 32 time steps and 48 pitch classes; values are binary. We sample random examples from the training set. We keep separate validation examples for inspection, although GANs do not have a validation loss as straightforward as a classifier.

**What this cell does:** prepares the data tensor and a batch of real excerpts. When using real chorales, inspect the active-note histogram and adapt the pitch range and time grid to the source.


In [ ]:
perm=np.random.permutation(len(rolls)); cut=max(1,int(.85*len(rolls)))
train_np,val_np=rolls[perm[:cut]],rolls[perm[cut:]]
train=torch.tensor(train_np,device=device)
LATENT=64; BATCH=32

def real_batch():
    ids=torch.randint(0,len(train),(BATCH,),device=device)
    return train[ids]
real=real_batch()
print("source:",data_source,"train:",train.shape,"batch:",real.shape,"active notes per piece:",real.sum((1,2)).float().mean().item())


## 3. Generator and critic

The generator receives a random vector `z` and maps it to a pianoroll matrix. Sigmoid produces note-activation probabilities. The critic returns one unbounded score per example, not a probability passed through sigmoid. Spectral normalization constrains the scale of the linear layers and can improve stability.

**What this cell does:** defines a small MLP generator and a convolutional critic. A MuseGAN variant with separate track generators and temporal modules would be a next step.


In [ ]:
from torch.nn.utils import spectral_norm

class Generator(nn.Module):
    def __init__(self,z_dim=LATENT):
        super().__init__(); self.net=nn.Sequential(
            nn.Linear(z_dim,256),nn.BatchNorm1d(256),nn.LeakyReLU(.2),
            nn.Linear(256,512),nn.BatchNorm1d(512),nn.LeakyReLU(.2),
            nn.Linear(512,STEPS*N_PITCH),nn.Sigmoid())
    def forward(self,z): return self.net(z).view(-1,1,STEPS,N_PITCH)

class Critic(nn.Module):
    def __init__(self):
        super().__init__(); self.net=nn.Sequential(
            spectral_norm(nn.Conv2d(1,32,4,2,1)),nn.LeakyReLU(.2),
            spectral_norm(nn.Conv2d(32,64,4,2,1)),nn.LeakyReLU(.2),
            nn.Flatten(),spectral_norm(nn.Linear(64*8*12,1)))
    def forward(self,x): return self.net(x).squeeze(1)

G=Generator().to(device); D=Critic().to(device)
print("G params:",sum(p.numel() for p in G.parameters()),"D params:",sum(p.numel() for p in D.parameters()))
print("generated batch shape:",G(torch.randn(4,LATENT,device=device)).shape)


## 4. Hinge objective and alternating updates

For the critic, hinge loss is $E[\max(0,1-D(x))]+E[\max(0,1+D(G(z)))]$. The generator minimizes $-E[D(G(z))]$. `detach()` during the critic step prevents the generator from being updated; during the generator step, the critic parameters are frozen, but gradients still flow through its computations to the generator input.

**What this cell does:** runs a few short training epochs. We limit the number of steps and model size so the demonstration does not train for hours. GAN losses do not have to decrease monotonically and should not be interpreted like supervised accuracy.


In [ ]:
optG=torch.optim.Adam(G.parameters(),lr=2e-4,betas=(0.0,.9))
optD=torch.optim.Adam(D.parameters(),lr=2e-4,betas=(0.0,.9))
EPOCHS=12; STEPS_PER_EPOCH=20; d_hist=[]; g_hist=[]
for epoch in range(EPOCHS):
    dl=[]; gl=[]
    for _ in range(STEPS_PER_EPOCH):
# 1) Critic: push real and fake samples to opposite sides of the margin.
        for p in D.parameters(): p.requires_grad_(True)
        xr=real_batch().unsqueeze(1); z=torch.randn(BATCH,LATENT,device=device)
        xf=G(z).detach(); dreal=D(xr); dfake=D(xf)
        lossD=F.relu(1-dreal).mean()+F.relu(1+dfake).mean()
        optD.zero_grad(set_to_none=True); lossD.backward(); optD.step()
        # 2) Generator: increase the critic score for generated samples.
        for p in D.parameters(): p.requires_grad_(False)
        z=torch.randn(BATCH,LATENT,device=device); fake=G(z)
        lossG=-D(fake).mean()
        optG.zero_grad(set_to_none=True); lossG.backward(); optG.step()
        dl.append(lossD.item()); gl.append(lossG.item())
    d_hist.append(float(np.mean(dl))); g_hist.append(float(np.mean(gl)))
    print(f"epoch {epoch+1:02d}/{EPOCHS} | D hinge={d_hist[-1]:.3f} | G={g_hist[-1]:.3f}")


## 5. Sample inspection, activity, and diversity

It is useful to evaluate both the structure of individual pianorolls and the differences between samples. The binary threshold can be chosen for visualization; it is not a universal MIDI decoder. We measure the average note count and mean pairwise XOR distance as simple indicators of activity and diversity.

**What this cell does:** generates a small grid of samples, compares them with a real batch, and computes simple statistics. Similar plots or near-zero diversity may indicate mode collapse, although these metrics do not evaluate harmony.


In [ ]:
@torch.no_grad()
def sample(n=8,threshold=.5):
    G.eval(); x=G(torch.randn(n,LATENT,device=device)).squeeze(1)
    G.train(); return (x>threshold).float().cpu().numpy()

fake=sample(8); real_show=real[:4].cpu().numpy()
fig,axes=plt.subplots(2,4,figsize=(12,5))
for i,ax in enumerate(axes.flat):
    arr=fake[i] if i<4 else real_show[i-4]
    ax.imshow(arr.T,aspect="auto",origin="lower",cmap="magma"); ax.set_title("fake" if i<4 else "real")
    ax.set_xlabel("time"); ax.set_ylabel("pitch")
plt.tight_layout(); plt.show()
flat=fake.reshape(len(fake),-1)
pairs=[np.mean(flat[i]!=flat[j]) for i in range(len(flat)) for j in range(i+1,len(flat))]
print("mean fake activity:",round(float(fake.sum((1,2)).mean()),1),"notes/piece")
print("mean pairwise XOR distance:",round(float(np.mean(pairs)),3))


## 6. Loss curves and training interpretation

In GANs, the generator and critic curves are coupled; a single value does not directly represent quality. If the critic dominates too quickly, the generator may receive a weak or unstable signal. If the generator produces only one family of outputs, consider more diverse data, architectural changes, and regularization.

**What this cell does:** plots both objective functions and considers them alongside visual and statistical sample checks. Evaluate results in several ways, including listening after MIDI conversion.


In [ ]:
fig,ax=plt.subplots(figsize=(7,3)); ax.plot(d_hist,label="krytyk hinge"); ax.plot(g_hist,label="generator")
ax.set_xlabel("epoch"); ax.set_ylabel("loss (different scales)"); ax.grid(); ax.legend(); plt.show()


## Key takeaways

- MuseGAN belongs to the family of adversarial models: the generator creates pianorolls from latent noise, and the critic learns to distinguish samples from the data.
- Hinge loss and spectral normalization are practical stabilizing components for a small demonstration, but they do not guarantee convergence.
- The pianoroll representation preserves simultaneous notes but loses many details of MIDI performance.
- GAN loss is not a simple quality metric. Evaluate activity, diversity, musical structure, validation examples, and listening results.
- This variant is inspired by MuseGAN, not a faithful reproduction: a fuller model uses a multi-track representation, shared codes, and temporal modules.
- Further work: a MIDI parser and the real JSB split, separate voice tracks, key/tempo control, gradient regularization, checkpointing, and MIDI export for listening to generated excerpts.